#Importing Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Loading the Data

In [ ]:
df=pd.read_csv('/content/Sample data (1).xlsx - Sheet1.csv')
df

#Basic Information

In [ ]:
#Shape of the Dataset
df.shape

In [ ]:
#Top 5 records
df.head(5)

In [ ]:
#Last 5 records
df.tail(5)

In [ ]:
#Data Types
df.info()

In [ ]:
#Statistical Summary
df.describe()

In [ ]:
#Number of Missing Values
print('The number of missing values is: ', df.isnull().sum())

In [ ]:
#Number of Duplicate Values
print('The number of duplicate values is: ', df.duplicated().sum())

#Data Preprocessing

In [ ]:
df['Discount Band'] = df['Discount Band'].fillna('None')
df

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y')

In [ ]:
df['Discount Rate'] = np.where(df['Gross Sales'] > 0, df['Discounts'] / df['Gross Sales'], 0)
df['Profit Margin'] = np.where(df[' Sales'] > 0, df['Profit'] / df[' Sales'], 0)
df.describe()

#Explorartory Data Analysis

In [ ]:
seg_summary = df.groupby('Segment').agg(
    Total_Sales=(' Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Margin=('Profit Margin', 'mean'),
    Units_Sold=('Units Sold', 'sum'),
    Orders=(' Sales', 'count')
).sort_values('Total_Sales', ascending=False)
seg_summary

In [ ]:
seg_summary['Total_Sales'].sort_values().plot(kind='barh', figsize=(7,4.5), color='#2E5B8A', title='Total Sales by Segment')
plt.xlabel('Sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
country_summary = df.groupby('Country').agg(
    Total_Sales=(' Sales', 'sum'), Total_Profit=('Profit', 'sum'), Avg_Margin=('Profit Margin', 'mean')
).sort_values('Total_Sales', ascending=False)
country_summary

In [ ]:
product_summary = df.groupby('Product').agg(
    Total_Sales=(' Sales', 'sum'), Total_Profit=('Profit', 'sum'),
    Units_Sold=('Units Sold', 'sum'), Avg_Margin=('Profit Margin', 'mean')
).sort_values('Total_Sales', ascending=False)
product_summary

In [ ]:
discount_summary = df.groupby('Discount Band').agg(
    Total_Sales=(' Sales', 'sum'), Total_Profit=('Profit', 'sum'), Avg_Discount_Rate=('Discount Rate', 'mean')
).sort_values('Total_Sales', ascending=False)
discount_summary

In [ ]:
monthly = df.groupby(df['Date'].dt.to_period('M')).agg(
    Total_Sales=(' Sales', 'sum'), Total_Profit=('Profit', 'sum')).reset_index()
monthly['Date'] = monthly['Date'].astype(str)

plt.figure(figsize=(9,4.5))
plt.plot(monthly['Date'], monthly['Total_Sales'], marker='o', label='Sales')
plt.plot(monthly['Date'], monthly['Total_Profit'], marker='o', label='Profit')
plt.xticks(rotation=60)
plt.legend()
plt.title('Monthly Sales & Profit Trend')
plt.tight_layout()
plt.show()

###Correlation Heatmap

In [ ]:
num_cols = ['Units Sold','Manufacturing Price','Sale Price','Gross Sales','Discounts',' Sales','COGS','Profit','Discount Rate']
plt.figure(figsize=(8,6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

#Regression Model

We predict `Sales` from variables known *before* a sale closes (units, pricing, segment, country, product, discount band, month)

In [ ]:
cat_features = ['Segment', 'Country', 'Product', 'Discount Band']
num_features = ['Units Sold', 'Sale Price', 'Manufacturing Price', 'Month Number']
target = ' Sales'

#Define Features and Target Variable

In [ ]:
X = df[cat_features + num_features]
y = df[target]

#Train and Test Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Encoding Data

In [ ]:
preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)], remainder='passthrough')

#Model Building

In [ ]:
results = {}
for name, model in [('Linear Regression', LinearRegression()),
                     ('Random Forest', RandomForestRegressor(n_estimators=200, random_state=42))]:
    pipe = Pipeline([('prep', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results[name] = {'r2': r2, 'mae': mae, 'rmse': rmse, 'pipe': pipe, 'pred': pred}
    print(f"{name}: R2={r2:.4f}  MAE={mae:,.0f}  RMSE={rmse:,.0f}")

In [ ]:
best_model = max(results, key=lambda k: results[k]['r2'])
best = results[best_model]
print('Best model:', best_model)

plt.figure(figsize=(6,6))
plt.scatter(y_test, best['pred'], alpha=0.5)
lims = [0, max(y_test.max(), best['pred'].max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual Sales'); plt.ylabel('Predicted Sales')
plt.title(f'Actual vs Predicted Sales ({best_name})')
plt.tight_layout()
plt.show()

In [ ]:
rf_pipe = results['Random Forest']['pipe']
ohe = rf_pipe.named_steps['prep'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(cat_features))
all_names = cat_names + num_features
importances = rf_pipe.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': all_names, 'importance': importances}).sort_values('importance', ascending=False).head(12)

plt.figure(figsize=(7,5))
plt.barh(imp_df['feature'][::-1], imp_df['importance'][::-1])
plt.title('Top Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()
imp_df

- Random Forest reaches R² ≈ 0.98 on held-out data (vs. ≈0.76 for plain Linear Regression) — it captures non-linear interactions between price, volume and segment/product mix.
- Units Sold and Sale Price dominate feature importance, as expected, but Segment/Product/Country add real predictive lift over price × units alone.
- MAE of roughly $11k on deals averaging ~$170K in sales is a strong result for pipeline forecasting.

#Findings and Recommendation

1. Government is your biggest customer. It brings in $52.5M, 44% of all sales, at a 22% margin.

- Keep these customers happy and renew their contracts.


2. Enterprise loses money. It sold $19.6M but lost $614K, and 58 of its 100 deals were unprofitable.

- Raise prices or cut discounts for Enterprise.

3. Channel Partners are very profitable but small. They make a 73% margin on only $1.8M in sales.

- Try to grow this channel.

4. Big discounts hurt profit. High-discount deals give away about 12.5% of the price, against 2.4% for low ones. Their profit margin is about 9%, versus about 18% for low-discount deals.

- Require manager approval for large discounts.

5. Sales go up and down by month. October is strong in both years, and June and December 2014 are also high.

- Plan stock and marketing before the busy months.

6. Paseo is the top product at $33.0M. Amarilla has the best margin at 15.9%, and Carretera has the lowest sales and a below-average margin.

- Push Paseo and Amarilla, and check Carretera's prices and costs.

7. The model can predict sales well (R² = 0.98).
- Use it to forecast revenue from planned units and prices.